# TD MAUP - M2 IGAST

## Comprendre le MAUP sur un cas pratique

Nous allons illustrer le MAUP avec la consommation d'espaces en France métropolitaine, à cinq maillages.

Les données viennent du Cerema. Il s'agit des surfaces d'espaces naturels, agricoles et forestiers (ENAF) consommées entre le 1er janvier 2011 et le 1er janvier 2025, par commune, en hectares.

L'indicateur du TD est la **part du territoire consommée en espaces naturels, agricoles et forestiers entre 2011 et 2025 (%)**. Dans la suite, on l'appelle plus simplement *taux de consommation d'espaces* (colonne `txartif`).

C'est un flux sur 14 ans, pas la part des sols artificialisés à une date. Une commune qui a beaucoup consommé n'est donc pas forcément « plus artificialisée » qu'une autre. Le [Cerema](https://doc-datafoncier.cerema.fr/doc/guide/conso-enaf/definition-de-la-consommation-despace) détaille la différence entre consommation d'espaces et artificialisation.

Chaque réponse s'appuie sur au moins un chiffre tiré de vos résultats, puis en donne une interprétation et une limite.

## Avant la séance

Le notebook se lit sur GitHub, mais il ne s'y exécute pas. Il s'ouvre dans Google Colab ou dans un environnement Python local. Les consignes pour installer cet environnement avec conda sont dans le [README du dépôt](https://github.com/fbxyz/M2_IGAST/blob/master/README.md#installation-et-utilisation-de-jupyterlab). Les fichiers de données sont aussi dans le dossier `data` du dépôt, à télécharger si la connexion est lente.

---

# 1 - Chargement et contrôle des données

Les cinq maillages regroupent les mêmes 34 735 communes.

| Maille | Fichier | Identifiant | Surface de la maille (ha) |
|:--|:--|:--|:--|
| Communes | `MAUP_COM_ARTIF_3857.geojson` | `insee_com` | `surface_com` |
| EPCI | `MAUP_EPCI_ARTIF_3857.geojson` | `insee_epci` | `surface_epci` |
| Départements | `MAUP_DEP_ARTIF_3857.geojson` | `insee_dep` | `surface_dept` |
| Anciennes régions (avant 2016) | `MAUP_OLDREG_ARTIF_3857.geojson` | `insee_old_reg` | `surface_oldreg` |
| Nouvelles régions (depuis 2016) | `MAUP_REG_ARTIF_3857.geojson` | `insee_reg` | `surface_reg` |

Tous les fichiers contiennent aussi `surface_artif` (surface d'ENAF consommée entre 2011 et 2025, en hectares), `pop11` et `pop22` (population en 2011 et en 2022, Insee).

Le nom de la colonne de surface change d'une maille à l'autre. Le taux se calcule avec toute la précision disponible, `round()` ne sert qu'à l'affichage.

## 1.1 - Communes : *com_artif*

Exemple pour les communes. La fonction `charger()` fait la même chose pour n'importe quelle maille.

In [ ]:
# On utilise geopandas pour charger des géométries dans un GeoDataFrame
# https://geopandas.org/
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify as mc
import numpy as np

sns.set_theme(style="whitegrid")

URL = "https://github.com/fbxyz/M2_IGAST/raw/master/data/MAUP_{}_ARTIF_3857.geojson"
LEG = "Part du territoire consommée 2011-2025 (%)"

# Charger le fichier GeoJSON des communes
com_artif = gpd.read_file(URL.format("COM"))

# On change la projection vers Lambert-93 (EPSG:2154)
com_artif = com_artif.to_crs(epsg=2154)

# Taux de consommation d'espaces 2011-2025, en pleine précision (pas de round ici)
com_artif['txartif'] = 100 * com_artif['surface_artif'] / com_artif['surface_com']

print(com_artif.head())

# La même chose pour n'importe quelle maille
MAILLES = {'Communes': ('COM', 'surface_com'), 'EPCI': ('EPCI', 'surface_epci'),
           'Départements': ('DEP', 'surface_dept'), 'Anciennes régions': ('OLDREG', 'surface_oldreg'),
           'Nouvelles régions': ('REG', 'surface_reg')}


def charger(nom):
    """Lit une maille, la reprojette en Lambert-93 et calcule le taux à partir des surfaces sommées."""
    fichier, surface = MAILLES[nom]
    gdf = gpd.read_file(URL.format(fichier)).to_crs(epsg=2154)
    gdf['txartif'] = 100 * gdf['surface_artif'] / gdf[surface]
    return gdf

## 1.2 - Les autres mailles

> `📝` Chargez les EPCI, les départements, les anciennes et les nouvelles régions dans *epci_artif*, *dep_artif*, *oldreg_artif* et *reg_artif*, avec le taux dans une colonne *txartif*

In [ ]:
# Avec la fonction charger() : epci_artif = charger('EPCI'), puis les autres mailles
# Sans elle : lire le fichier, reprojeter, calculer le taux avec la bonne colonne de surface

## 1.3 - Contrôles

> `📝` Exécutez ces contrôles. Les surfaces sont-elles toutes positives ? Manque-t-il des valeurs ? Les sommes sont-elles identiques d'une maille à l'autre ?

In [ ]:
mailles = {'Communes': com_artif, 'EPCI': epci_artif, 'Départements': dep_artif,
           'Anciennes régions': oldreg_artif, 'Nouvelles régions': reg_artif}

controles = []
for nom, gdf in mailles.items():
    surface = MAILLES[nom][1]
    controles.append({'Maille': nom, 'Unités': len(gdf),
                      'Surfaces nulles ou négatives': int((gdf[surface] <= 0).sum()),
                      'Valeurs manquantes': int(gdf[[surface, 'surface_artif', 'pop11', 'pop22']].isna().sum().sum()),
                      'Surface consommée (ha)': round(gdf['surface_artif'].sum()),
                      'Surface totale (ha)': round(gdf[surface].sum()),
                      'Population 2022': int(gdf['pop22'].sum())})
pd.DataFrame(controles)

---

# 2 - Cartographie des données

Les cinq cartes utilisent les mêmes bornes, la même palette, la même emprise et la même légende. Seul le maillage change d'une carte à l'autre.

Les bornes (0,25, 0,5, 1 et 2 %, puis le maximum communal de 35,7 %) sont des valeurs arrondies qui doublent d'une classe à l'autre, de part et d'autre du taux de la France métropolitaine (0,53 %). La dernière classe couvre toutes les valeurs, quelle que soit la maille.

> `📝` Consultez l'aide de mapclassify si besoin : <https://pysal.org/mapclassify/>

## 2.1 - Carte par communes

In [ ]:
# Bornes communes à toutes les mailles (limite supérieure incluse)
BORNES = [0.25, 0.5, 1, 2, 36]
assert all(g['txartif'].max() <= BORNES[-1] for g in mailles.values()), "une valeur dépasse la dernière borne"


def carte(gdf, nom, epaisseur=0.1):
    """Carte choroplèthe d'une maille, avec les bornes communes."""
    # ÉTAPE 1 : une figure, mêmes dimensions pour toutes les mailles
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    # ÉTAPE 2 : la carte, avec des classes définies par l'utilisateur
    gdf.plot(column='txartif', cmap='OrRd', scheme='user_defined', classification_kwds={'bins': BORNES, 'lowest': 0},
             legend=True, ax=ax, edgecolor='grey', linewidth=epaisseur,
             legend_kwds={'title': LEG, 'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'frameon': True, 'fmt': '{:.2f}'})
    # ÉTAPE 3 : titre, source, sans axes
    ax.set_title(f"Consommation d'espaces 2011-2025, {nom}", fontsize=14, fontweight='bold')
    ax.axis('off')
    ax.text(0.02, 0.02, 'Source : Cerema, 2025', transform=ax.transAxes, fontsize=8, style='italic')
    plt.tight_layout()
    plt.show()


carte(com_artif, "communes")

## 2.2 - Les autres mailles

> `📝` Cartographiez les EPCI, les départements, les anciennes et les nouvelles régions avec les mêmes bornes

In [ ]:
# Avec la fonction carte() : carte(epci_artif, "EPCI", epaisseur=0.3), puis les autres mailles

## 2.3 - Questions

> `📝` **Question 1 :** Comment la carte change-t-elle de la commune à la région ?
>
> `📝` **Question 2 :** Les zones qui ont le plus consommé restent-elles les mêmes à chaque maille ?
>
> `📝` **Question 3 :** Quelle maille vous semble la plus pertinente pour étudier la consommation d'espaces ? Pourquoi ?

**Vos réponses ici :**

1.
2.
3.

---

# 3 - Analyse univariée

En utilisant les acquis des cours précédents, décrivez la distribution du taux à chaque maille.

## 3.1 - Tableau comparatif

> `📝` Calculez pour chaque maille la moyenne, la médiane, l'écart-type, le minimum et le maximum du taux, puis rassemblez-les dans un tableau

In [ ]:
def resume(nom, gdf):
    """Statistiques descriptives du taux pour une maille."""
    t = gdf['txartif']
    return {'Maille': nom, 'Unités': len(gdf), 'Moyenne': t.mean(), 'Médiane': t.median(),
            'Écart-type': t.std(), 'Minimum': t.min(), 'Maximum': t.max()}


resume('Communes', com_artif)

In [ ]:
# TODO: appliquer resume() à chaque maille du dictionnaire mailles, puis en faire un DataFrame

## 3.2 - Boxplots côte à côte

Le graphique coupe l'axe à 5 % pour que les boîtes restent lisibles. Le nombre d'unités au-delà est écrit au-dessus de chaque boîte.

In [ ]:
# Un DataFrame au format « long » pour seaborn : une ligne par unité, sa maille et son taux
df_long = pd.concat({nom: gdf['txartif'] for nom, gdf in mailles.items()}, names=['Maille', 'index']).reset_index(name=LEG)

PLAFOND = 5
fig, ax = plt.subplots(1, 1, figsize=(12, 7))
sns.boxplot(data=df_long, x='Maille', y=LEG, ax=ax, hue='Maille', palette='Set2', legend=False)
ax.set_ylim(0, PLAFOND)
for i, (nom, gdf) in enumerate(mailles.items()):
    n = int((gdf['txartif'] > PLAFOND).sum())
    ax.text(i, PLAFOND * 0.97, f"{n} unité(s) au-delà de {PLAFOND} %", ha='center', va='top', fontsize=9, color='firebrick')
ax.set_title("Consommation d'espaces 2011-2025 selon le maillage", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.3 - Questions

> `📝` **Question 1 :** Comment évoluent la moyenne et la médiane selon la maille ?
>
> `📝` **Question 2 :** Comment évolue la dispersion selon la maille ? Pourquoi ?
>
> `📝` **Question 3 :** Quel lien faites-vous entre ces observations et le MAUP ?

**Vos réponses ici :**

1.
2.
3.

---

# 4 - Analyse bivariée

La consommation d'espaces suit-elle la croissance de la population ? Les fichiers contiennent la population de 2011 et de 2022 (`pop11`, `pop22`). Vous allez mesurer leur corrélation R à chaque maille.

Les périodes diffèrent. La croissance porte sur 2011-2022, la consommation sur 2011-2025.

> `📝` Calculez la croissance de la population entre 2011 et 2022 (en %) pour chaque maille
>
> `📝` Calculez le coefficient de corrélation R entre cette croissance et le taux de consommation d'espaces

## 4.1 - Communes

Exemple pour les communes. R est calculé sur toutes les communes, mais le cadre du graphique en coupe quelques-unes. Leur nombre est indiqué sur la figure.

In [ ]:
# Quelques communes n'avaient aucun habitant en 2011 : leur croissance n'est pas définie
d = com_artif[com_artif['pop11'] > 0].copy()
d['croissance'] = 100 * (d['pop22'] - d['pop11']) / d['pop11']

r = d['txartif'].corr(d['croissance'])
print(f"Communes : R = {r:.2f} ({len(d)} unités)")

XLIM, YLIM = (-40, 80), (0, 5)
hors = ~(d['croissance'].between(*XLIM) & d['txartif'].between(*YLIM))

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(d['croissance'], d['txartif'], s=4, alpha=0.4, color='firebrick')
ax.set_title(f"Communes, R = {r:.2f}", fontsize=14)
ax.set_xlabel('Croissance de la population 2011-2022 (%)')
ax.set_ylabel(LEG)
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.text(0.98, 0.97, f"{hors.sum()} communes hors du cadre, comptées dans R", transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='firebrick')
plt.tight_layout()
plt.show()

## 4.2 - Les autres mailles

> `📝` Calculez R pour les EPCI, les départements, les anciennes et les nouvelles régions
>
> `📝` Tracez le nuage de points d'une autre maille

In [ ]:
# TODO: pour chaque maille, calculer la croissance puis R (une boucle sur le dictionnaire mailles)
# TODO: nuage de points pour une autre maille

## 4.3 - Questions

> `📝` **Question 1 :** Comment évolue R de la commune à la région ?
>
> `📝` **Question 2 :** Que pèse un point quand il ne reste que 13 régions ? Un R proche de zéro montre-t-il l'absence de relation ?
>
> `📝` **Question 3 :** Ce calcul permet-il de dire que la croissance démographique cause la consommation d'espaces ?

**Vos réponses ici :**

1.
2.
3.

---

# Tâche individuelle

Chacun rédige seul une courte réponse (une demi-page) sur un indicateur qui n'a pas été vu en TD, par exemple la part des logements vacants par commune. Quelle représentation choisir ? Quelles classes proposer ? Que pourrait modifier un passage à une autre maille (EPCI, département), dans la carte comme dans les conclusions ?

---

# Pour aller plus loin

1. **L'autocorrélation spatiale.** Comment évolue-t-elle selon la maille ? (indice de Moran avec `esda` de PySAL)
2. **Le changement de découpage de 2016.** Comparez le rang de chaque ancienne région avec celui de la région qui l'a remplacée. Qu'est-ce qui change en même temps que les limites ?
3. **Le lissage spatial.** Remplacez le zonage par une grille régulière et un rayon de voisinage, puis comparez plusieurs rayons

N'oubliez pas de consulter la présentation du cours pour réviser les notions du MAUP.